# Hyperparameter Optimization & AutoML

Companion notebook for the [Hyperparameter Optimization lesson](https://ml-viz-ruby.vercel.app/courses/optimization-ml/05-hyperparameter-optimization).

We empirically reproduce the classic result that **random search beats grid search** when only a
few hyperparameters matter, and implement **Successive Halving** to see how early stopping finds a
good config with far less total compute. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## 1 — A toy validation surface where only 1 of 2 hyperparameters matters

The score depends strongly on h1 and barely on h2 — the common real-world case. The best h1 is 0.5.

In [ ]:
def val_score(h1, h2):
    return np.exp(-((h1 - 0.5)**2) / 0.02) + 0.03 * np.cos(10 * h2)   # h2 barely matters

best_possible = val_score(0.5, 0.0)
print(f'best achievable score ~ {best_possible:.3f} (at h1=0.5)')

## 2 — Random search vs grid search at equal budget

Both get 16 trials. Grid spends them on a 4×4 lattice (only 4 distinct h1 values). Random tries 16
distinct h1 values — so it samples the dimension that matters far more finely.

In [ ]:
budget = 16
g = np.linspace(0, 1, 4)
grid_best = max(val_score(a, b) for a in g for b in g)

rand_scores = []
for trial in range(200):
    r = np.random.default_rng(trial)
    rand_best = max(val_score(r.random(), r.random()) for _ in range(budget))
    rand_scores.append(rand_best)

print(f'grid search best   (4x4):     {grid_best:.3f}')
print(f'random search best (avg of 200 runs): {np.mean(rand_scores):.3f}')
print(f'random search wins {100*np.mean(np.array(rand_scores) > grid_best):.0f}% of the time')

## 3 — Successive Halving: stop the losers early

Start many configs with a small budget, keep the top half, double their budget, repeat. Most configs
die cheap; compute concentrates on the survivors. We use noisy early estimates that improve with more
budget (more epochs = less noise).

In [ ]:
def noisy_eval(h1, h2, budget, seed):
    r = np.random.default_rng(seed)
    return val_score(h1, h2) + r.normal(0, 0.3 / np.sqrt(budget))   # noise shrinks with budget

def successive_halving(n_configs=16, min_budget=1):
    configs = [(rng.random(), rng.random(), i) for i in range(n_configs)]
    budget = min_budget; total = 0
    while len(configs) > 1:
        scored = [(noisy_eval(h1, h2, budget, sid), (h1, h2, sid)) for h1, h2, sid in configs]
        total += len(configs) * budget                           # compute spent this rung
        scored.sort(reverse=True)
        configs = [c for _, c in scored[:max(1, len(scored)//2)]]  # keep top half
        budget *= 2
    return configs[0], total

(best, tot) = successive_halving()
print(f'Successive Halving picked h1={best[0]:.2f} (target 0.5), total compute = {tot} budget-units')
print(f'Running all 16 configs to the max budget would cost {16 * 16} budget-units.')

## ✏️ Your turn

**Exercise.** Implement `random_search(score_fn, budget, seed)` returning the best score over `budget`
random (h1, h2) draws in [0,1]², and `keep_top_half(scored)` returning the better half of a list of
`(score, config)` pairs (the core pruning step of Successive Halving).

In [ ]:
def random_search(score_fn, budget, seed=0):
    r = np.random.default_rng(seed)
    # TODO(you): draw `budget` random (h1,h2) in [0,1]^2, return the best score_fn value
    return ...

def keep_top_half(scored):
    # TODO(you): scored is a list of (score, config); return the top half by score (at least 1)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
s = random_search(val_score, 50, seed=1)
assert s <= best_possible + 1e-9                              # can't beat the true max
assert random_search(val_score, 200, 1) >= random_search(val_score, 4, 1)   # more budget, no worse
kept = keep_top_half([(0.1, 'a'), (0.9, 'b'), (0.5, 'c'), (0.3, 'd')])
assert set(c for _, c in kept) == {'b', 'c'}                  # the two highest scores
assert len(keep_top_half([(0.5, 'x')])) == 1                  # never drop the last one
print('\u2713 random search and top-half pruning are correct')

<details>
<summary>Solution</summary>

```python
def random_search(score_fn, budget, seed=0):
    r = np.random.default_rng(seed)
    return max(score_fn(r.random(), r.random()) for _ in range(budget))

def keep_top_half(scored):
    scored = sorted(scored, reverse=True)
    return scored[:max(1, len(scored)//2)]
```

Random search wins because it samples the *important* hyperparameter at many distinct values instead
of wasting a grid on irrelevant ones; Successive Halving wins by not paying full price for configs
that are clearly losing.

</details>